In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings
import numpy as np
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

from gridr.chain.grid_resampling_chain import basic_grid_resampling_chain
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
from chain_notebook_utils import (
    write_array, create_grid, create_star_polygon, plot_grid_on_image,
)

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# File paths shared across the chain tutorial series.
RASTER_IN          = "./grid_resampling_chain_raster_in.tif"
GRID_IN_F64        = "./grid_resampling_chain_grid_in_f64.tif"
output_raster_path = "./grid_resampling_chain_output_raster.tif"
output_mask_path   = "./grid_resampling_chain_output_mask.tif"

# Grid parameters shared across the chain tutorial series.
nrow, ncol = 50, 40
origin_pos  = np.array((0.3, 0.2))
origin_node = np.array((0., 0.))
v_row_y, v_row_x = 5.2, 1.2
v_col_y, v_col_x = -2.7, 7.1

grid_row_f64, grid_col_f64 = create_grid(
    nrow, ncol, origin_pos, origin_node,
    v_row_y, v_row_x, v_col_y, v_col_x, dtype=np.float64,
)

# Hybrid input creation: ensure the source raster and grid file exist on disk.
if not os.path.exists(RASTER_IN):
    write_array(mandrill, dtype=mandrill.dtype, fileout=RASTER_IN)
if not os.path.exists(GRID_IN_F64):
    write_array(np.array([grid_row_f64, grid_col_f64]),
                dtype=np.float64, fileout=GRID_IN_F64)

# Default output-dataset open arguments (resolution-dependent overrides
# are applied per-notebook below).
grid_mask_in_path        = "./grid_resampling_chain_grid_mask_in_u8_1_0.tif"
grid_mask_in_path_i8     = "./grid_resampling_chain_grid_mask_in_i8_0_m10.tif"
raster_mask_in_path_u8   = "./grid_resampling_chain_raster_mask_in_u8_1_0.tif"


# Getting started with the GridR resampling chain

This tutorial introduces `basic_grid_resampling_chain`, the file-oriented wrapper around `array_grid_resampling`. The core function works on in-memory NumPy arrays; the chain layer manages all I/O against rasterio datasets and processes the output in memory-efficient strips and tiles.

This first notebook shows the minimal call: open a source raster, open a grid file, open a destination file, and let the chain do the rest.

**What you'll learn**

- How to prepare on-disk inputs for the chain (raster + grid)
- How to open input/output rasterio datasets in a `with` block
- How to call `basic_grid_resampling_chain` with minimal arguments
- How `grid_resolution` controls the output size

## Setting things up

The chain reads from rasterio `DatasetReader` and writes to `DatasetWriter` objects opened by the caller. Before calling it we need:

1. A **source raster** on disk -- here, the mandrill image.
2. A **grid file** on disk -- a two-band GeoTIFF where band 1 holds row   coordinates and band 2 holds column coordinates.

Both files are created above by the tutorial setup cell using the `write_array` and `create_grid` helpers from `chain_notebook_utils`.

The grid is an affine-parameterised regular grid of shape `(50, 40)` mapped onto the source raster's coordinate frame. The figure below overlays its nodes on the source image.

In [ ]:
plot_grid_on_image(
    1.5, grid_row_f64, grid_col_f64, (1, 1),
    (mandrill.shape[1], mandrill.shape[2]),
    mask=None, win=None, raster_image=mandrill[0],
    prefix="grid_on_image",
)

## First call: identity resolution

At `grid_resolution=(1, 1)` the output has exactly as many pixels as there are grid nodes (`50 × 40`). The output dataset must be opened with matching dimensions.

In [ ]:
output_shape_1_1 = grid_row_f64.shape

raster_out_open_args_1_1 = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape_1_1[0], "width": output_shape_1_1[1], "count": 1,
}

with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args_1_1) as array_out_ds:

    basic_grid_resampling_chain(
        grid_ds              = grid_in_ds,
        grid_row_coords_band = 1,
        grid_col_coords_band = 2,
        grid_resolution      = (1, 1),
        array_src_ds         = array_src_ds,
        array_src_bands      = 1,
        array_out_ds         = array_out_ds,
        interp               = "cubic",
        nodata_out           = 0,
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as ds:
    plot_im(ds.read(1), prefix="resampling_1_1")

The output is small because the grid subsamples the source raster heavily (each node is several source pixels apart). Out-of-domain nodes are filled with `nodata_out`.

## Zooming with `grid_resolution`

`grid_resolution` inserts additional output samples between adjacent grid nodes by bilinear interpolation of the node coordinates. Setting it to `(8, 8)` multiplies the output dimensions by 8 in each direction. The output shape must be sized accordingly using `grid_full_resolution_shape`.

In [ ]:
from gridr.core.grid.grid_commons import grid_full_resolution_shape

grid_resolution = (8, 8)

output_shape = grid_full_resolution_shape(
    shape=grid_row_f64.shape, resolution=grid_resolution,
)

raster_out_open_args = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape[0], "width": output_shape[1], "count": 1,
}
print(f"output shape: {output_shape}")

In [ ]:
with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds:

    basic_grid_resampling_chain(
        grid_ds              = grid_in_ds,
        grid_row_coords_band = 1,
        grid_col_coords_band = 2,
        grid_resolution      = grid_resolution,
        array_src_ds         = array_src_ds,
        array_src_bands      = 1,
        array_out_ds         = array_out_ds,
        interp               = "cubic",
        nodata_out           = 0,
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as ds:
    plot_im(ds.read(1), prefix="resampling_8_8")

The next notebooks build on this minimal call by adding masks, geometry constraints, coordinate shifts and I/O controls one at a time.